# Phase 1: Audio Data Exploration and Feature Extraction

Welcome to the first phase of the **Emotion Speech Recognition** project utilizing the **RAVDESS** (Ryerson Audio-Visual Database of Emotional Speech and Song) dataset.

### Objectives of this Notebook:
1. **Environment Setup:** Configure the workspace, link our custom modular scripts (`src/data_loader.py`), and define global data paths.
2. **Raw Data Exploration:** Load a sample `.wav` file to visualize its raw waveform and inspect the time-domain amplitude.
3. **Auditory Augmentation Check:** Audibly verify the effects of our programmatic noise injection and pitch-shifting to ensure the synthetic data sounds realistic and useful.
4. **Feature Verification:** Extract and visualize the Mel-Frequency Cepstral Coefficients (MFCCs) alongside their 1st and 2nd derivatives (Deltas). This matrix will act as the "image" input for our CNN.
5. **Batch Processing:** Trigger the global pipeline to convert all raw `.wav` files into serialized, machine-learning-ready `.npy` tensors.

In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES & CONFIGURE PATHS
# ==========================================
import os # Needed for path functions like os.path.join, os.path.basename, etc.
import sys # Needed to add the src folder to the hidden list of folder locations that Python searches.
import glob # Needed for recursively fetching files from nested directories.
import numpy as np
import librosa # The core library for audio analysis and feature extraction.
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd # Enables playable audio widgets directly inside the Jupyter Notebook.

# Enable autoreload. This is crucial during development: any changes saved to .py files 
# in the `src` folder will be automatically imported into this notebook without restarting the kernel.
%load_ext autoreload
%autoreload 2

# Connect to the src folder by adding the parent directory to sys.path
sys.path.append(os.path.abspath(".."))

# Import the core extraction function we built previously in data_loader.py
from src.data_loader import extract_and_serialize_features

# Define global paths for the data source where the raw audio files are kept 
# and where the final processed .npy files will be saved.
DATA_SOURCE = os.path.join("..", "data", "raw")
DATA_DESTINATION = os.path.join("..", "data", "processed")

print(f"Data Source mapped to: {DATA_SOURCE}")
print(f"Data Destination mapped to: {DATA_DESTINATION}")

### 2. Loading a Sample File and Visualizing the Waveform
Before extracting complex frequency features for all files, it is vital to perform a sanity check on a single sample.

Here, we scan the `raw` directory, grab the first audio file we find, and visualize its raw waveform. The waveform gives us a strict time-domain view of the speech amplitude (how loud the speaker is at any given fraction of a second).

In [ ]:
# ==========================================
# 2. WAVEFORM VISUALIZATION
# ==========================================
# Find all .wav files in the raw data directory recursively
sample_files = glob.glob(os.path.join(DATA_SOURCE, "**", "*.wav"), recursive=True) # Contain every file path for all .wav files in the RAVDESS dataset.

if sample_files:
    # Select the first file as a sample for our exploration
    sample_audio_path = sample_files[0]
    print(f"Exploring File: {os.path.basename(sample_audio_path)}")
    
    # Load the audio using librosa. 
    # We force a sampling rate (sr) of 16000 Hz (standard for speech modeling). 
    # We also cap the duration at 4 seconds to ensure we drop extended trailing silence 
    # and maintain consistent tensor footprints downstream.
    sr_raw = 16000
    y_raw, _ = librosa.load(sample_audio_path, sr=sr_raw, duration=4)

    # Summary of the Data loaded into the Array using librosa.load 
    total_samples = len(y_raw) # To know the number the number of data points in the array
    duration_seconds = total_samples / sr_raw # Using the sample rate and number of data points to determine the seconds of audio loaded. (Sampling Rate * Seconds = Total Array Item)
    
    # The Audio Math Deduction 
    print("--- DIGITAL AUDIO MATH ---")
    print(f"1. Total Array Items (Samples): {total_samples:,}")
    print(f"2. Sampling Rate (Snapshots per second): {sr_raw:,} Hz")
    print(f"3. Deduced Duration: {total_samples:,} / {sr_raw:,} = {duration_seconds:.2f} seconds\n")
    
   # Plot the Waveforms (Macro vs Micro)
    plt.figure(figsize=(14, 4))
    
    # 1. Full Waveform
    plt.subplot(1, 2, 1)
    librosa.display.waveshow(y_raw, sr=sr_raw, color="#3498db")
    plt.title(f"Full Waveform ({duration_seconds:.2f}s)", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Time (seconds)", fontsize=12, labelpad=10)
    plt.ylabel("Amplitude", fontsize=12, labelpad=10)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    
    # 2. Zoomed Waveform
    plt.subplot(1, 2, 2)
    librosa.display.waveshow(y_raw, sr=sr_raw, color="#e74c3c") 
    plt.title("Zoomed (0.8s to 2.5s)", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Time (seconds)", fontsize=12, labelpad=10)
    plt.ylabel("Amplitude", fontsize=12, labelpad=10)
    plt.xlim(0.8, 2.5)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    
    plt.tight_layout()
    plt.show()

else:
    print("No .wav files found! Please check your DATA_SOURCE path and ensure the RAVDESS dataset is present.")

### 3. Auditory Verification of Data Augmentation
Data augmentation is a critical tool for preventing neural network overfitting, especially on smaller, pristine datasets like RAVDESS. 

By adding **White Noise** and **Pitch Shifting**, we simulate real-world variations such as staticky microphones, background hum, and natural vocal cord depth differences. Below, we apply the exact mathematical formulas derived from our `data_loader.py` module to physically hear the effect they have on the audio signal before the neural network ever sees it.

In [ ]:
# ==========================================
# 3. AUGMENTATION AUDIO PLAYBACK
# ==========================================
if sample_files:
    # Create an interactive playback widget for the clean, unadulterated audio
    print("1. Playing ORIGINAL Audio:")
    display(ipd.Audio(y_raw, rate=sr_raw))
    
    print("\n2. Playing WHITE NOISE Augmented Audio:")
    # We multiply by the max amplitude (np.amax) to ensure the noise amplitude naturally scales 
    # up or down depending on whether the speaker is yelling or whispering. 
    # This creates a realistic "staticky" background hum.
    # This is the exact same formula used in the data_loader.py script for the white noise augmentation.
    noise_amp = 0.005 * np.random.uniform() * np.amax(y_raw)
    noisy_audio = y_raw + noise_amp * np.random.normal(size=y_raw.shape[0])
    display(ipd.Audio(noisy_audio, rate=sr_raw))
    
    print("\n3. Playing PITCH SHIFTED Augmented Audio (-1 Semitone):")
    # We shift it down by 1 semitone (-1) to make the effect clearly audible.
    # This accurately mimics the acoustic profile of a speaker with a deeper, richer voice.
    pitched_audio = librosa.effects.pitch_shift(y=y_raw, sr=sr_raw, n_steps=-1)
    display(ipd.Audio(pitched_audio, rate=sr_raw))

### 4. Feature Extraction (MFCCs & Deltas)
Neural networks struggle to identify complex emotional patterns from raw 1D amplitude graphs. Instead, we convert the audio into a 2D matrix called **Mel-Frequency Cepstral Coefficients (MFCCs)**.

MFCCs represent the short-term power spectrum of a sound, mathematically warped to match human auditory perception (the Mel scale). However, emotion isn't just about static frequencies—it's about how those frequencies change over time.

Therefore, we compute the base MFCCs, but we also compute their 1st and 2nd order derivatives (**Deltas** and **Delta-Deltas**). If base MFCCs are the "position" of the voice, Deltas are the "velocity" and Delta-Deltas are the "acceleration." Stacking these together gives our CRNN immense temporal context.

In [ ]:
# ==========================================
# 4. MFCC EXTRACTION & VISUALIZATION
# ==========================================
if sample_files:
    # 1. Extract 20 base MFCCs (captures the static envelope of the vocal tract)
    mfccs = librosa.feature.mfcc(y=y_raw, sr=sr_raw, n_mfcc=20)
    
    # 2. Compute 1st order deltas (captures the velocity/trajectory of MFCC changes)
    delta_mfccs = librosa.feature.delta(mfccs)
    
    # 3. Compute 2nd order deltas (captures the acceleration of MFCC changes)
    delta2_mfccs = librosa.feature.delta(mfccs, order=2)
    
    # Concatenate vertically (axis=0) to create a stacked feature matrix of shape (60, time_steps)
    combined_features = np.concatenate((mfccs, delta_mfccs, delta2_mfccs), axis=0)
    
    # Plot the combined MFCC feature map as a heatmap (spectrogram-style)
    plt.figure(figsize=(14, 6))
    # We use x_axis='time' so Librosa automatically maps the array columns to physical seconds
    librosa.display.specshow(combined_features, sr=sr_raw, x_axis='time', cmap='magma')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Combined MFCC Features + Deltas (The "Image" for our CRNN)', fontsize=14, pad=15)
    plt.ylabel('Feature Coefficients (0-19: Base, 20-39: $\Delta$, 40-59: $\Delta\Delta$)', fontsize=12)
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print(f"Combined Feature Matrix Shape: {combined_features.shape}")
    print("Notice how the rows represent feature coefficients and columns represent time steps.")

### 5. Executing the Complete Batch Extraction Pipeline
Now that we understand the process conceptually and visually, it is time to execute the heavy lifting across the entire dataset. We will invoke our custom `extract_and_serialize_features` function from `src/data_loader.py`.

**What this automated step does:**
1. Recursively scans the `raw` directory for all RAVDESS `.wav` files.
2. Parses the strict RAVDESS file naming convention to extract the correct Emotion and Actor IDs.
3. Applies White Noise and Pitch Shift augmentations (**only** to training actors 1-20 to avoid test data leakage).
4. Extracts the 60-row combined MFCC feature matrices.
5. Pads or truncates them to exactly 125 time-steps to ensure uniform input geometry for the neural network.
6. Serializes the final `X_features.npy`, `y_labels.npy`, and `actor_ids.npy` binary matrices to the `processed` data directory.
7. Confirm that the vinary matrices are saved properly by loading them
8. Check the number of occurence for each class for cases of data imbalance

In [ ]:
# ==========================================
# 5. FULL DATASET EXTRACTION & SERIALIZATION
# ==========================================

# Note: Because this iterates through hundreds of files, applies real-time acoustic 
# transformations, and performs Fourier transforms, this might take a few minutes.

print("Initiating global feature extraction pipeline...")

# We pass our predefined paths to the loader function.
# Default extraction parameters apply: max_pad_len=125, n_mfcc=20, sr=16000
extract_and_serialize_features(data_dir=DATA_SOURCE, save_dir=DATA_DESTINATION)

print("\nNotebook 1 execution complete! The feature matrices are serialized on disk and ready for Notebook 2.")

In [ ]:
# Load the saved data to prove it worked
X = np.load(os.path.join(DATA_DESTINATION, "X_features.npy"))
y = np.load(os.path.join(DATA_DESTINATION, "y_labels.npy"))

print(f"Verified Final X Shape: {X.shape}")
print(f"Verified Final y Shape: {y.shape}")

In [ ]:
# Class Distribution of each Emotion
# Count the occurrences of each emotion
classes, counts = np.unique(y, return_counts=True)

# Plot Class Distribution
plt.figure(figsize=(10, 5))
plt.bar(classes, counts, color='#2ecc71', edgecolor='black')
plt.title("RAVDESS Emotion Class Distribution", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Emotion", fontsize=12, labelpad=10)
plt.ylabel("Number of Audio Samples", fontsize=12, labelpad=10)
plt.xticks(rotation=45, fontsize=11)
plt.yticks(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# EDA - Comparative MFCCs to see Visual diffrences in their MFCCs(Happy vs. Sad vs. Neutral)

# 1. Safely grab the FIRST occurrence of each emotion. 
# Because of our data_loader loop, the first appended file is ALWAYS the clean original!
happy_idx = np.where(y == 'happy')[0][0]
sad_idx = np.where(y == 'sad')[0][0]
neutral_idx = np.where(y == 'neutral')[0][0]

def plot_advanced_features(X_data, emotion_title):
    """Plots the 60-row feature matrix split into 3 distinct acoustic subplots."""
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    
    # Base MFCCs (Rows 0-19)
    img1 = axes[0].imshow(X_data[0:20], aspect='auto', origin='lower', cmap='coolwarm')
    axes[0].set_title(f"{emotion_title} - Base MFCCs")
    fig.colorbar(img1, ax=axes[0], format='%+2.0f')
    
    # Velocity Deltas (Rows 20-39)
    img2 = axes[1].imshow(X_data[20:40], aspect='auto', origin='lower', cmap='coolwarm')
    axes[1].set_title(f"{emotion_title} - Velocity (Deltas)")
    fig.colorbar(img2, ax=axes[1], format='%+2.0f')
    
    # Acceleration Deltas (Rows 40-59)
    img3 = axes[2].imshow(X_data[40:60], aspect='auto', origin='lower', cmap='coolwarm')
    axes[2].set_title(f"{emotion_title} - Acceleration (Delta-Deltas)")
    fig.colorbar(img3, ax=axes[2], format='%+2.0f')
    
    plt.xlabel('Time Frames')
    plt.tight_layout()
    plt.show()

# Generate the new visual comparisons
plot_advanced_features(X[happy_idx], "Happy")
plot_advanced_features(X[sad_idx], "Sad")
plot_advanced_features(X[neutral_idx], "Neutral")